In [ ]:
# ==============================================================================
# ĐỒ ÁN HỒI QUY TUYẾN TÍNH - CHỦ ĐỀ 1: DỰ ĐOÁN HIỆU NĂNG CPU (PASSMARK SCORE)
# ==============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D

sns.set_theme(style="whitegrid")

# ==============================================================================
# PHẦN 1: NHIỆM VỤ CỦA KIÊN - TIỀN XỬ LÝ, MERGE DATA & TRỰC QUAN HÓA (Task 1 & 2)
# ==============================================================================
# 1. Nạp 2 file dữ liệu
df_main = pd.read_csv('Final_CPU_Data_Cleaned.csv')
df_bench = pd.read_csv('CPU_benchmark_v4.csv')

# 2. Ghép nối (Merge) dữ liệu lấy điểm Benchmark
# Khớp cột 'name' (file gốc) với cột 'cpuName' (file benchmark)
df_merged = pd.merge(df_main, df_bench[['cpuName', 'cpuMark']], 
                     left_on='name', right_on='cpuName', how='inner')

# CHỐT BIẾN MỤC TIÊU: cpuMark (Điểm Benchmark thực tế)
target_col = 'cpuMark' 

# 3. Làm sạch dữ liệu
df_clean = df_merged.dropna(subset=['speed_ghz', target_col]).copy()
df_clean = df_clean[(df_clean['speed_ghz'] > 0) & (df_clean[target_col] > 0)]

print(f"Số lượng CPU hợp lệ sau khi ghép nối và làm sạch: {len(df_clean)} dòng.")

# (Tùy chọn) Lưu file data mới đã ghép nối để cả nhóm dùng chung
df_clean.to_csv('Merged_CPU_Data_With_Score.csv', index=False)

# 4. Vẽ biểu đồ Scatter Plot (Đã cập nhật tiêu đề theo ý kiến của Sang)
plt.figure(figsize=(8, 5))
sns.regplot(x=df_clean['speed_ghz'], y=df_clean[target_col], 
            scatter_kws={'alpha':0.5, 'color':'blue', 'edgecolor': 'w', 'linewidths': 0.5}, 
            line_kws={'color':'red', 'linewidth': 2, 'linestyle': '--'})
plt.title('CPU Clock Speed vs PassMark CPU Mark: Linear Regression', fontsize=14, fontweight='bold')
plt.xlabel('Clock Speed (GHz)', fontsize=12)
plt.ylabel('PassMark CPU Mark (Điểm hiệu năng)', fontsize=12)
plt.tight_layout()
plt.show()

# 5. Đóng gói biến X và y
X_raw = df_clean[['speed_ghz']].values
y_vec = df_clean[[target_col]].values.reshape(-1, 1)
m = len(X_raw)

# Chuẩn hóa Z-score cho biến X
X_mean = np.mean(X_raw)
X_std = np.std(X_raw)
X_scaled = (X_raw - X_mean) / X_std

# Thêm cột bias vào ma trận X
X_b = np.c_[np.ones((m, 1)), X_scaled]

# ==============================================================================
# PHẦN 2: NHIỆM VỤ CỦA XUÂN HẠ - THUẬT TOÁN (Task 3, 4, 5)
# ==============================================================================
# Nhiệm vụ 3: Hàm tính Chi phí (Cost Function)
def compute_cost(X, y, theta):
    predictions = X.dot(theta)
    errors = predictions - y
    return (1 / (2 * m)) * np.sum(np.square(errors))

# Nhiệm vụ 4: Thuật toán Gradient Descent (Lưu trữ 501 trạng thái)
def gradient_descent(X, y, theta, learning_rate, iterations):
    cost_history = np.zeros(iterations + 1)
    theta_history = []
    
    # Lưu trạng thái 1 (Khởi tạo)
    cost_history[0] = compute_cost(X, y, theta)
    theta_history.append(theta.copy())
    
    # Lưu trạng thái 2 đến 501 (Vòng lặp)
    for i in range(iterations):
        predictions = X.dot(theta)
        errors = predictions - y
        gradients = (1 / m) * X.T.dot(errors)
        theta = theta - learning_rate * gradients
        
        cost_history[i + 1] = compute_cost(X, y, theta)
        theta_history.append(theta.copy())
        
    return theta, cost_history, theta_history

# Chạy Gradient Descent
alpha = 0.1
iterations = 500
theta_initial = np.zeros((2, 1))

theta_gd, cost_history, theta_history = gradient_descent(X_b, y_vec, theta_initial, alpha, iterations)

# Nhiệm vụ 5: Hàm Normal Equation (Nghiệm Giải tích)
def normal_equation(X, y):
    return np.linalg.inv(X.T.dot(X)).dot(X.T).dot(y)

theta_analytic = normal_equation(X_b, y_vec)

print("\n--- KẾT QUẢ THUẬT TOÁN ---")
print(f"Hệ số chặn (theta_0): {theta_gd[0][0]:.4f}")
print(f"Hệ số góc (theta_1):  {theta_gd[1][0]:.4f}")
sai_so = np.sum(np.abs(theta_gd - theta_analytic))
print(f"=> Độ chênh lệch giữa GD và Giải tích: {sai_so:.8f}")

# ==============================================================================
# PHẦN 3: NHIỆM VỤ CỦA ĐẠT SANG - ĐÁNH GIÁ & TRỰC QUAN HÓA 3D (Task 6, 7, 8)
# ==============================================================================
# Nhiệm vụ 8: Đánh giá mô hình (Đã bổ sung Pearson Correlation)
r_matrix = np.corrcoef(X_raw.flatten(), y_vec.flatten())
pearson_r = r_matrix[0, 1]

y_pred = X_b.dot(theta_gd)
mse = np.mean(np.square(y_pred - y_vec))
rmse = np.sqrt(mse)

ss_total = np.sum(np.square(y_vec - np.mean(y_vec)))
ss_res = np.sum(np.square(y_vec - y_pred))
r2_score = 1 - (ss_res / ss_total)

print("\n--- ĐÁNH GIÁ CHẤT LƯỢNG MÔ HÌNH ---")
print(f"Pearson Correlation (r): {pearson_r:.4f}")
print(f"R^2 Score:               {r2_score:.4f}")
print(f"MSE:                     {mse:.4f}")
print(f"RMSE:                    {rmse:.4f}")

# Nhiệm vụ 6 & 7: Vẽ 3D Cost Surface & Contour Plot kèm Quỹ đạo hội tụ
theta0_hist = [t[0][0] for t in theta_history]
theta1_hist = [t[1][0] for t in theta_history]

# Tính toán biên độ không gian dựa trên giá trị theta mới
t0_margin = np.abs(theta_gd[0][0]) * 0.5 if theta_gd[0][0] != 0 else 2000
t1_margin = np.abs(theta_gd[1][0]) * 0.5 if theta_gd[1][0] != 0 else 2000

theta0_range = np.linspace(theta_gd[0][0] - t0_margin, theta_gd[0][0] + t0_margin, 50)
theta1_range = np.linspace(theta_gd[1][0] - t1_margin, theta_gd[1][0] + t1_margin, 50)
T0, T1 = np.meshgrid(theta0_range, theta1_range)
J_vals = np.zeros((len(theta0_range), len(theta1_range)))

for i in range(len(theta0_range)):
    for j in range(len(theta1_range)):
        t = np.array([[theta0_range[i]], [theta1_range[j]]])
        J_vals[i, j] = compute_cost(X_b, y_vec, t)

J_vals = J_vals.T

fig = plt.figure(figsize=(16, 7))
plt.style.use('default') 

# Đồ thị 1: 3D Cost Surface
ax1 = fig.add_subplot(121, projection='3d')
ax1.plot_surface(T0, T1, J_vals, cmap='plasma', alpha=0.5, edgecolor='none')
ax1.plot(theta0_hist, theta1_hist, cost_history, color='red', linewidth=2, marker='o', markersize=3, label='Optimization Trajectory')
ax1.scatter(theta_gd[0][0], theta_gd[1][0], cost_history[-1], color='black', s=100, zorder=5, label='Global Minimum')

ax1.set_title('3D Cost Surface (PassMark)', fontsize=14, fontweight='bold', pad=20)
ax1.set_xlabel('Theta 0 (Intercept)', labelpad=10)
ax1.set_ylabel('Theta 1 (Slope)', labelpad=10)
ax1.set_zlabel('Cost J', labelpad=10)
ax1.legend()

# Đồ thị 2: Contour Plot
ax2 = fig.add_subplot(122)
levels = np.logspace(np.log10(np.min(J_vals)), np.log10(np.max(J_vals)), 30)
ax2.contour(T0, T1, J_vals, levels=levels, cmap='plasma')
ax2.plot(theta0_hist, theta1_hist, color='red', linewidth=2, marker='o', markersize=3, label='GD Path')
ax2.scatter(theta_gd[0][0], theta_gd[1][0], color='black', marker='X', s=150, linewidths=3, label='Global Minimum', zorder=5)

ax2.set_title('Contour Plot (PassMark)', fontsize=14, fontweight='bold', pad=20)
ax2.set_xlabel('Theta 0 (Intercept)')
ax2.set_ylabel('Theta 1 (Slope)')
ax2.legend()

plt.tight_layout()
plt.show()